# Clases y objetos · Patrones y principios SOLID

Cómo se comportan los objetos al pasarlos a funciones, cómo se comprueban sus tipos y cómo resuelve Python la herencia múltiple, seguido de varios patrones de diseño (objetos invocables, *singleton*, máquina de estados, observador) y de los cinco principios SOLID, cada uno con un diseño que lo incumple, otro que lo cumple y comprobaciones que muestran la diferencia. Termina con una máquina expendedora que da el cambio con las monedas que tiene. Ideas de [retosdeprogramacion.com](https://retosdeprogramacion.com/roadmap) y [PYnative](https://pynative.com/python-object-oriented-programming-oop-exercise/).

## 1. Paso de argumentos por asignación

En Python, el parámetro de una función es un nombre nuevo para el mismo objeto que se pasa: no se copia el valor ni se pasa la variable. Comprobar con `is` que la función recibe el mismo objeto, que mutarlo se ve fuera y reasignarlo no, que `+=` muta una lista pero crea una tupla nueva, que una función no puede intercambiar las variables del llamador (y cómo hacerlo devolviendo una tupla) y la trampa del valor por defecto mutable.

In [1]:
def es_el_mismo(parametro, original):
    return parametro is original


def anadir_isla(islas, isla):
    islas.append(isla)


def reasignar(islas):
    islas = ["Atlántida"]
    return islas


def ampliar(secuencia, extra):
    secuencia += extra
    return secuencia


canarias = ["Tenerife", "Gran Canaria"]
print("¿Recibe el mismo objeto?", es_el_mismo(canarias, canarias))
anadir_isla(canarias, "La Palma")
print("Tras mutar:   ", canarias)
reasignar(canarias)
print("Tras reasignar:", canarias)
assert canarias == ["Tenerife", "Gran Canaria", "La Palma"]

lista, tupla = [1, 2], (1, 2)
print("ampliar(lista):", ampliar(lista, [3]), "-> fuera:", lista)
print("ampliar(tupla):", ampliar(tupla, (3,)), "-> fuera:", tupla)
assert lista == [1, 2, 3] and tupla == (1, 2)

¿Recibe el mismo objeto? True
Tras mutar:    ['Tenerife', 'Gran Canaria', 'La Palma']
Tras reasignar: ['Tenerife', 'Gran Canaria', 'La Palma']
ampliar(lista): [1, 2, 3] -> fuera: [1, 2, 3]
ampliar(tupla): (1, 2, 3) -> fuera: (1, 2)


In [2]:
def intercambiar_mal(a, b):
    a, b = b, a


def intercambiar(a, b):
    return b, a


x, y = "ida", "vuelta"
intercambiar_mal(x, y)
print("intercambiar_mal:", x, y)
x, y = intercambiar(x, y)
print("intercambiar:    ", x, y)
assert (x, y) == ("vuelta", "ida")


def apuntar_mal(nombre, lista=[]):
    lista.append(nombre)
    return lista


def apuntar(nombre, lista=None):
    lista = [] if lista is None else lista
    lista.append(nombre)
    return lista


print("Valor por defecto mutable:", apuntar_mal("Ana"), apuntar_mal("Beni"))
print("Con None:                 ", apuntar("Ana"), apuntar("Beni"))
assert apuntar_mal.__defaults__ == (["Ana", "Beni"],) and apuntar("Carla") == ["Carla"]

intercambiar_mal: ida vuelta
intercambiar:     vuelta ida
Valor por defecto mutable: ['Ana', 'Beni'] ['Ana', 'Beni']
Con None:                  ['Ana'] ['Beni']


## 2. `type()`, `isinstance()` e `issubclass()`

Crear una jerarquía de transportes: `Transporte`, de la que heredan `Terrestre` y `Maritimo`; de `Terrestre`, `Guagua` y `Tranvia`, y de `Maritimo`, `Ferry`. Mostrar en una tabla, para un objeto de cada clase, su `type`, si `type(x) is Terrestre` y si `isinstance(x, Terrestre)`, y comprobar `issubclass` con clases y con tuplas de clases. Escribir una función que liste todas las subclases (no solo las directas que da `__subclasses__`) y ver por qué `True` es un `int`.

In [3]:
class Transporte: ...
class Terrestre(Transporte): ...
class Maritimo(Transporte): ...
class Guagua(Terrestre): ...
class Tranvia(Terrestre): ...
class Ferry(Maritimo): ...


objetos = [Transporte(), Terrestre(), Guagua(), Tranvia(), Ferry()]
print(f"{'type':<11}{'type is Terrestre':>19}{'isinstance Terrestre':>22}{'isinstance (Maritimo, Tranvia)':>32}")
for o in objetos:
    print(f"{type(o).__name__:<11}{type(o) is Terrestre!s:>19}{isinstance(o, Terrestre)!s:>22}"
          f"{isinstance(o, (Maritimo, Tranvia))!s:>32}")

assert issubclass(Guagua, Transporte) and issubclass(Transporte, Transporte)
assert not issubclass(Ferry, Terrestre) and issubclass(Ferry, (Terrestre, Maritimo))


def todas_subclases(clase):
    resultado = []
    for hija in clase.__subclasses__():
        resultado += [hija, *todas_subclases(hija)]
    return resultado


print("Directas:", [c.__name__ for c in Transporte.__subclasses__()])
print("Todas:   ", [c.__name__ for c in todas_subclases(Transporte)])
assert set(todas_subclases(Transporte)) == {c for c in (Terrestre, Maritimo, Guagua, Tranvia, Ferry)}

print("bool es subclase de int:", issubclass(bool, int), "| True + True =", True + True,
      "| type(True) is int:", type(True) is int)
assert isinstance(True, int) and bool.__mro__ == (bool, int, object)

type         type is Terrestre  isinstance Terrestre  isinstance (Maritimo, Tranvia)
Transporte               False                 False                           False
Terrestre                 True                  True                           False
Guagua                   False                  True                           False
Tranvia                  False                  True                            True
Ferry                    False                 False                            True
Directas: ['Terrestre', 'Maritimo']
Todas:    ['Terrestre', 'Guagua', 'Tranvia', 'Maritimo', 'Ferry']
bool es subclase de int: True | True + True = 2 | type(True) is int: False


## 3. Herencia múltiple, `super()` cooperativo y el diamante

Modelar un `Smartphone` que hereda de `Telefono` y de `Camara`, que a su vez heredan de `Dispositivo` (un diamante). Con `super()` cooperativo y `**kwargs`, cada constructor recoge sus argumentos y pasa el resto. Mostrar el MRO con `__mro__`, comprobar que el constructor de `Dispositivo` se ejecuta una sola vez y compararlo con llamar a los constructores de los padres a mano, donde se ejecuta dos veces. Ver también qué ocurre si se pide un orden imposible.

In [4]:
traza = []


class Dispositivo:
    def __init__(self, marca, **kwargs):
        super().__init__(**kwargs)
        self.marca = marca
        traza.append("Dispositivo")

    def describir(self):
        return [f"marca {self.marca}"]


class Telefono(Dispositivo):
    def __init__(self, numero, **kwargs):
        super().__init__(**kwargs)
        self.numero = numero
        traza.append("Telefono")

    def describir(self):
        return [f"número {self.numero}", *super().describir()]


class Camara(Dispositivo):
    def __init__(self, megapixeles, **kwargs):
        super().__init__(**kwargs)
        self.megapixeles = megapixeles
        traza.append("Camara")

    def describir(self):
        return [f"{self.megapixeles} MP", *super().describir()]


class Smartphone(Telefono, Camara):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        traza.append("Smartphone")


movil = Smartphone(marca="Guanche", numero="928 000 000", megapixeles=48)
print("MRO:", " -> ".join(c.__name__ for c in Smartphone.__mro__))
print("Orden de los constructores:", traza)
print("describir():", movil.describir())
assert traza == ["Dispositivo", "Camara", "Telefono", "Smartphone"]
assert movil.describir() == ["número 928 000 000", "48 MP", "marca Guanche"]

MRO: Smartphone -> Telefono -> Camara -> Dispositivo -> object
Orden de los constructores: ['Dispositivo', 'Camara', 'Telefono', 'Smartphone']
describir(): ['número 928 000 000', '48 MP', 'marca Guanche']


In [5]:
contador = {"Base": 0}


class Base:
    def __init__(self):
        contador["Base"] += 1


class Izquierda(Base):
    def __init__(self):
        Base.__init__(self)


class Derecha(Base):
    def __init__(self):
        Base.__init__(self)


class Abajo(Izquierda, Derecha):
    def __init__(self):
        Izquierda.__init__(self)
        Derecha.__init__(self)


Abajo()
print("Llamando a los padres a mano, Base.__init__ se ejecuta", contador["Base"], "veces")
assert contador["Base"] == 2

try:
    class Imposible(Dispositivo, Telefono): ...
except TypeError as e:
    print("TypeError:", e)

Llamando a los padres a mano, Base.__init__ se ejecuta 2 veces
TypeError: Cannot create a consistent method resolution order (MRO) for bases Dispositivo, Telefono


## 4. Objetos invocables con `__call__`

Crear una clase `Multiplicador` cuyas instancias se usen como funciones (`doble(21)`), también con `map` y `sorted`. Después crear `ContarLlamadas`, una clase que sirva como decorador: guarda la función, cuenta cuántas veces se llama y conserva su nombre con `functools.update_wrapper`. Usarla con Fibonacci recursivo y comprobar que `fib(n)` hace $2F_{n+1} - 1$ llamadas.

In [6]:
from functools import update_wrapper


class Multiplicador:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor

    def __repr__(self):
        return f"Multiplicador({self.factor})"


doble, triple = Multiplicador(2), Multiplicador(3)
print(doble, "->", doble(21), "| callable:", callable(doble), "| map:", list(map(triple, [1, 2, 3])))
print("Ordenar por triple con signo:", sorted([4, -1, 3, -5], key=Multiplicador(-3)))
assert doble(21) == 42 and not callable(42)


class ContarLlamadas:
    def __init__(self, funcion):
        update_wrapper(self, funcion)
        self.funcion = funcion
        self.llamadas = 0

    def __call__(self, *args, **kwargs):
        self.llamadas += 1
        return self.funcion(*args, **kwargs)


@ContarLlamadas
def fib(n):
    """Fibonacci recursivo sin memoria."""
    return n if n < 2 else fib(n - 1) + fib(n - 2)


def fibonacci_iterativo(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


for n in (5, 10, 15):
    fib.llamadas = 0
    valor = fib(n)
    print(f"fib({n:>2}) = {valor:>3} con {fib.llamadas:>4} llamadas")
    assert valor == fibonacci_iterativo(n) and fib.llamadas == 2 * fibonacci_iterativo(n + 1) - 1
print(fib.__name__, "-", fib.__doc__)

Multiplicador(2) -> 42 | callable: True | map: [3, 6, 9]
Ordenar por triple con signo: [4, 3, -1, -5]
fib( 5) =   5 con   15 llamadas
fib(10) =  55 con  177 llamadas
fib(15) = 610 con 1973 llamadas
fib - Fibonacci recursivo sin memoria.


## 5. *Singleton*: una única sesión de usuario

Una aplicación debe tener una sola sesión de usuario activa. Implementarla con `__new__`, comprobando con `is` que dos «creaciones» devuelven el mismo objeto, y ver la trampa de añadir un `__init__`, que se vuelve a ejecutar en cada llamada y reinicia el estado. Implementar después una configuración global con una metaclase que intercepta la llamada a la clase, y comparar con un decorador de clase, que rompe `isinstance`.

In [7]:
class SesionUsuario:
    _instancia = None

    def __new__(cls, usuario):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.usuario = usuario
            cls._instancia.carrito = []
        return cls._instancia


s1 = SesionUsuario("tamara")
s1.carrito.append("mojo verde")
s2 = SesionUsuario("intruso")
print("s1 is s2:", s1 is s2, "| usuario:", s2.usuario, "| carrito:", s2.carrito)
assert s1 is s2 and s2.usuario == "tamara"


class SesionConInit:
    _instancia = None

    def __new__(cls, usuario):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
        return cls._instancia

    def __init__(self, usuario):
        self.usuario = usuario
        self.carrito = []


a = SesionConInit("tamara")
a.carrito.append("mojo verde")
b = SesionConInit("intruso")
print("Con __init__: a is b:", a is b, "| usuario:", a.usuario, "| carrito:", a.carrito)
assert a is b and a.carrito == []

s1 is s2: True | usuario: tamara | carrito: ['mojo verde']
Con __init__: a is b: True | usuario: intruso | carrito: []


In [8]:
from functools import wraps


class UnicaInstancia(type):
    _instancias = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instancias:
            cls._instancias[cls] = super().__call__(*args, **kwargs)
        return cls._instancias[cls]


class Configuracion(metaclass=UnicaInstancia):
    def __init__(self, idioma="es"):
        self.idioma = idioma


c1, c2 = Configuracion("es"), Configuracion("en")
print("Metaclase: c1 is c2:", c1 is c2, "| idioma:", c2.idioma, "| isinstance:", isinstance(c1, Configuracion))
assert c1 is c2 and c2.idioma == "es"


def unica(clase):
    instancias = {}

    @wraps(clase)
    def obtener(*args, **kwargs):
        if clase not in instancias:
            instancias[clase] = clase(*args, **kwargs)
        return instancias[clase]

    return obtener


@unica
class Registro:
    def __init__(self):
        self.lineas = []


assert Registro() is Registro()
try:
    isinstance(Registro(), Registro)
except TypeError as e:
    print("Decorador: Registro ya no es una clase ->", e)

Metaclase: c1 is c2: True | idioma: es | isinstance: True
Decorador: Registro ya no es una clase -> isinstance() arg 2 must be a type, a tuple of types, or a union


## 6. Máquina de estados de un pedido

Definir los estados de un pedido con `Enum` (pendiente, pagado, preparando, enviado, entregado, cancelado y devuelto) y un diccionario con las transiciones permitidas. La clase `Pedido` cambia de estado con `pasar_a`, que lanza `TransicionInvalida` si el cambio no está permitido y guarda el historial. Aplicar una secuencia de cambios a dos pedidos capturando los inválidos, y comprobar que los estados finales no tienen salida y que todos los estados se alcanzan desde *pendiente*.

In [9]:
from enum import Enum, auto


class Estado(Enum):
    PENDIENTE = auto()
    PAGADO = auto()
    PREPARANDO = auto()
    ENVIADO = auto()
    ENTREGADO = auto()
    CANCELADO = auto()
    DEVUELTO = auto()


TRANSICIONES = {
    Estado.PENDIENTE: {Estado.PAGADO, Estado.CANCELADO},
    Estado.PAGADO: {Estado.PREPARANDO, Estado.CANCELADO},
    Estado.PREPARANDO: {Estado.ENVIADO},
    Estado.ENVIADO: {Estado.ENTREGADO},
    Estado.ENTREGADO: {Estado.DEVUELTO},
    Estado.CANCELADO: set(),
    Estado.DEVUELTO: set(),
}


class TransicionInvalida(Exception):
    pass


class Pedido:
    def __init__(self, codigo):
        self.codigo = codigo
        self.estado = Estado.PENDIENTE
        self.historial = [self.estado]

    def pasar_a(self, nuevo):
        if nuevo not in TRANSICIONES[self.estado]:
            raise TransicionInvalida(f"{self.codigo}: no se puede pasar de {self.estado.name} a {nuevo.name}")
        self.estado = nuevo
        self.historial.append(nuevo)

    @property
    def terminado(self):
        return not TRANSICIONES[self.estado]


pedidos = {"P-101": Pedido("P-101"), "P-102": Pedido("P-102")}
operaciones = [("P-101", Estado.PAGADO), ("P-101", Estado.ENVIADO), ("P-101", Estado.PREPARANDO),
               ("P-102", Estado.CANCELADO), ("P-101", Estado.ENVIADO), ("P-102", Estado.PAGADO),
               ("P-101", Estado.ENTREGADO), ("P-101", Estado.CANCELADO)]
fallos = 0
for codigo, nuevo in operaciones:
    try:
        pedidos[codigo].pasar_a(nuevo)
        print(f"{codigo} -> {nuevo.name}")
    except TransicionInvalida as e:
        fallos += 1
        print("  rechazado:", e)
for p in pedidos.values():
    print(f"{p.codigo}: {' -> '.join(e.name for e in p.historial)} (terminado: {p.terminado})")
assert fallos == 3 and pedidos["P-101"].estado is Estado.ENTREGADO and pedidos["P-102"].terminado

P-101 -> PAGADO
  rechazado: P-101: no se puede pasar de PAGADO a ENVIADO
P-101 -> PREPARANDO
P-102 -> CANCELADO
P-101 -> ENVIADO
  rechazado: P-102: no se puede pasar de CANCELADO a PAGADO
P-101 -> ENTREGADO
  rechazado: P-101: no se puede pasar de ENTREGADO a CANCELADO
P-101: PENDIENTE -> PAGADO -> PREPARANDO -> ENVIADO -> ENTREGADO (terminado: False)
P-102: PENDIENTE -> CANCELADO (terminado: True)


In [10]:
assert set(TRANSICIONES) == set(Estado)
assert {e for e, salida in TRANSICIONES.items() if not salida} == {Estado.CANCELADO, Estado.DEVUELTO}

alcanzables, pendientes = {Estado.PENDIENTE}, [Estado.PENDIENTE]
while pendientes:
    for siguiente in TRANSICIONES[pendientes.pop()] - alcanzables:
        alcanzables.add(siguiente)
        pendientes.append(siguiente)
assert alcanzables == set(Estado)

for origen in Estado:
    for destino in Estado:
        p = Pedido("prueba")
        p.estado = origen
        try:
            p.pasar_a(destino)
            assert destino in TRANSICIONES[origen]
        except TransicionInvalida:
            assert destino not in TRANSICIONES[origen] and p.estado is origen
print("Transiciones válidas:", sum(map(len, TRANSICIONES.values())), "de", len(Estado) ** 2, "pares posibles")

Transiciones válidas: 7 de 49 pares posibles


## 7. Observador: avisos meteorológicos

Crear un `CentroAvisos` en el que los interesados se suscriben a un tema (`calima`, `oleaje`, `lluvia`) con cualquier invocable, y que al publicar un aviso llame a todos los suscriptores de ese tema. `suscribir` devuelve una función para darse de baja. Un suscriptor que falla no debe impedir que el resto reciba el aviso: capturar el error y contarlo. Suscribir una app móvil, un panel del aeropuerto y un registro, y comprobar quién recibe cada aviso.

In [11]:
from collections import defaultdict


class CentroAvisos:
    def __init__(self):
        self._suscriptores = defaultdict(list)
        self.errores = []

    def suscribir(self, tema, callback):
        self._suscriptores[tema].append(callback)
        return lambda: self._suscriptores[tema].remove(callback)

    def publicar(self, tema, mensaje):
        entregados = 0
        for callback in list(self._suscriptores[tema]):
            try:
                callback(tema, mensaje)
                entregados += 1
            except Exception as e:
                self.errores.append((tema, type(e).__name__))
        return entregados


class AppMovil:
    def __init__(self, usuario):
        self.usuario = usuario
        self.bandeja = []

    def recibir(self, tema, mensaje):
        self.bandeja.append(f"[{tema}] {mensaje}")


class PanelAeropuerto:
    def __init__(self):
        self.ultimo = {}

    def __call__(self, tema, mensaje):
        self.ultimo[tema] = mensaje


def pantalla_rota(tema, mensaje):
    raise RuntimeError("pantalla sin conexión")


centro = CentroAvisos()
app = AppMovil("yeray")
panel = PanelAeropuerto()
registro = []
baja_app_lluvia = centro.suscribir("lluvia", app.recibir)
centro.suscribir("calima", app.recibir)
for tema in ("calima", "oleaje"):
    centro.suscribir(tema, panel)
for tema in ("calima", "oleaje", "lluvia"):
    centro.suscribir(tema, lambda t, m: registro.append((t, m)))
centro.suscribir("oleaje", pantalla_rota)

avisos = [("calima", "Aviso amarillo en Lanzarote y Fuerteventura"), ("lluvia", "Chubascos en el norte de Tenerife"),
          ("oleaje", "Olas de 4 m en el norte de Gran Canaria")]
baja_app_lluvia()
for tema, mensaje in avisos:
    print(f"{tema:<7} entregado a {centro.publicar(tema, mensaje)} suscriptores")
print("App:", app.bandeja)
print("Panel:", panel.ultimo)
print("Errores:", centro.errores)
assert app.bandeja == ["[calima] Aviso amarillo en Lanzarote y Fuerteventura"]
assert set(panel.ultimo) == {"calima", "oleaje"} and len(registro) == 3
assert centro.errores == [("oleaje", "RuntimeError")]

calima  entregado a 3 suscriptores
lluvia  entregado a 1 suscriptores
oleaje  entregado a 2 suscriptores
App: ['[calima] Aviso amarillo en Lanzarote y Fuerteventura']
Panel: {'calima': 'Aviso amarillo en Lanzarote y Fuerteventura', 'oleaje': 'Olas de 4 m en el norte de Gran Canaria'}
Errores: [('oleaje', 'RuntimeError')]


## 8. SRP: responsabilidad única

*Principio de responsabilidad única*: una clase debe tener un solo motivo para cambiar. Escribir una `BibliotecaTodoEnUno` que preste y devuelva libros, los guarde en un fichero JSON y genere un informe, y separarla en `Prestamos` (reglas de préstamo), `RepositorioJSON` (persistencia en cualquier flujo de texto) e informes intercambiables. Comprobar que las reglas se prueban sin tocar el disco, que el repositorio guarda y recupera lo mismo con `io.StringIO` y que añadir un informe en CSV no obliga a tocar `Prestamos`.

In [12]:
import io
import json
from pathlib import Path


class BibliotecaTodoEnUno:
    def __init__(self, ruta):
        self.ruta = Path(ruta)
        self.prestamos = {}

    def prestar(self, titulo, socio):
        if titulo in self.prestamos:
            raise ValueError(f"«{titulo}» ya está prestado")
        self.prestamos[titulo] = socio
        self.ruta.write_text(json.dumps(self.prestamos, ensure_ascii=False), encoding="utf-8")

    def informe(self):
        return "\n".join(f"{t} -> {s}" for t, s in sorted(self.prestamos.items()))


vieja = BibliotecaTodoEnUno("biblioteca_srp.json")
vieja.prestar("Mararía", "Acerina")
print("Diseño acoplado: prestar un libro escribe en", vieja.ruta.name, "->", vieja.ruta.read_text(encoding="utf-8"))

Diseño acoplado: prestar un libro escribe en biblioteca_srp.json -> {"Mararía": "Acerina"}


In [13]:
class Prestamos:
    def __init__(self, maximo_por_socio=2):
        self.maximo_por_socio = maximo_por_socio
        self._prestados = {}

    def prestar(self, titulo, socio):
        if titulo in self._prestados:
            raise ValueError(f"«{titulo}» ya está prestado")
        if len(self.de(socio)) >= self.maximo_por_socio:
            raise ValueError(f"{socio} ya tiene {self.maximo_por_socio} libros")
        self._prestados[titulo] = socio

    def devolver(self, titulo):
        del self._prestados[titulo]

    def de(self, socio):
        return sorted(t for t, s in self._prestados.items() if s == socio)

    def todos(self):
        return dict(sorted(self._prestados.items()))


class RepositorioJSON:
    def guardar(self, prestamos, flujo):
        json.dump(prestamos.todos(), flujo, ensure_ascii=False)

    def cargar(self, flujo):
        prestamos = Prestamos()
        for titulo, socio in json.load(flujo).items():
            prestamos.prestar(titulo, socio)
        return prestamos


class InformeTexto:
    def generar(self, prestamos):
        return "\n".join(f"{t:<26}{s}" for t, s in prestamos.todos().items())


class InformeCSV:
    def generar(self, prestamos):
        return "\n".join(["titulo,socio", *(f"{t},{s}" for t, s in prestamos.todos().items())])


prestamos = Prestamos()
for titulo, socio in [("Mararía", "Acerina"), ("Viera y Clavijo", "Dácil"), ("Fortunata y Jacinta", "Acerina")]:
    prestamos.prestar(titulo, socio)
for intento in [("Mararía", "Dácil"), ("Cien años de soledad", "Acerina")]:
    try:
        prestamos.prestar(*intento)
    except ValueError as e:
        print("Rechazado:", e)

flujo = io.StringIO()
RepositorioJSON().guardar(prestamos, flujo)
flujo.seek(0)
recuperados = RepositorioJSON().cargar(flujo)
for informe in (InformeTexto(), InformeCSV()):
    print(f"--- {type(informe).__name__}\n{informe.generar(recuperados)}")

assert recuperados.todos() == prestamos.todos() and prestamos.de("Acerina") == ["Fortunata y Jacinta", "Mararía"]
assert not any(hasattr(Prestamos, m) for m in ("guardar", "cargar", "generar", "informe"))
assert {m for m in vars(BibliotecaTodoEnUno) if not m.startswith("_")} == {"prestar", "informe"}

Rechazado: «Mararía» ya está prestado
Rechazado: Acerina ya tiene 2 libros
--- InformeTexto
Fortunata y Jacinta       Acerina
Mararía                   Acerina
Viera y Clavijo           Dácil
--- InformeCSV
titulo,socio
Fortunata y Jacinta,Acerina
Mararía,Acerina
Viera y Clavijo,Dácil


## 9. OCP: abierto a la extensión, cerrado a la modificación

*Principio abierto/cerrado*: debe poder añadirse comportamiento sin modificar el código que ya funciona. Escribir una `CalculadoraRigida` con un `if/elif` por operación y comprobar que, para añadir la potencia, habría que editarla. Rediseñarla con una clase abstracta `Operacion` (símbolo y método `aplicar`) y una `Calculadora` que solo conoce las operaciones que se le registran. Añadir desde fuera la potencia, el módulo y la hipotenusa, y comprobar que la clase `Calculadora` no ha cambiado.

In [14]:
import math
import operator
from abc import ABC, abstractmethod


class CalculadoraRigida:
    def calcular(self, simbolo, a, b):
        if simbolo == "+":
            return a + b
        elif simbolo == "-":
            return a - b
        elif simbolo == "*":
            return a * b
        elif simbolo == "/":
            return a / b
        raise ValueError(f"operación desconocida: {simbolo}; hay que editar la clase")


try:
    CalculadoraRigida().calcular("**", 2, 10)
except ValueError as e:
    print("Rígida:", e)


class Operacion(ABC):
    simbolo: str

    @abstractmethod
    def aplicar(self, a, b): ...


class Calculadora:
    def __init__(self, *operaciones):
        self._operaciones = {}
        for op in operaciones:
            self.registrar(op)

    def registrar(self, operacion):
        self._operaciones[operacion.simbolo] = operacion

    def calcular(self, simbolo, a, b):
        try:
            return self._operaciones[simbolo].aplicar(a, b)
        except KeyError:
            raise ValueError(f"operación no registrada: {simbolo}") from None

    def simbolos(self):
        return sorted(self._operaciones)

Rígida: operación desconocida: **; hay que editar la clase


In [15]:
class Binaria(Operacion):
    def __init__(self, simbolo, funcion):
        self.simbolo, self.funcion = simbolo, funcion

    def aplicar(self, a, b):
        return self.funcion(a, b)


class Hipotenusa(Operacion):
    simbolo = "hip"

    def aplicar(self, a, b):
        return math.hypot(a, b)


calc = Calculadora(*(Binaria(s, f) for s, f in [("+", operator.add), ("-", operator.sub),
                                                 ("*", operator.mul), ("/", operator.truediv)]))
antes = dict(vars(Calculadora))
for nueva in (Binaria("**", operator.pow), Binaria("%", operator.mod), Hipotenusa()):
    calc.registrar(nueva)

expresiones = [("+", 7, 5), ("/", 9, 4), ("**", 2, 10), ("%", 17, 5), ("hip", 3, 4)]
for simbolo, a, b in expresiones:
    print(f"{a} {simbolo} {b} = {calc.calcular(simbolo, a, b)}")
print("Registradas:", calc.simbolos())
assert dict(vars(Calculadora)) == antes
assert calc.calcular("**", 2, 10) == 1024 and calc.calcular("hip", 3, 4) == 5
assert all(calc.calcular(s, 13, 4) == CalculadoraRigida().calcular(s, 13, 4) for s in "+-*/")
try:
    Operacion()
except TypeError as e:
    print("TypeError:", e)

7 + 5 = 12
9 / 4 = 2.25
2 ** 10 = 1024
17 % 5 = 2
3 hip 4 = 5.0
Registradas: ['%', '*', '**', '+', '-', '/', 'hip']
TypeError: Can't instantiate abstract class Operacion without an implementation for abstract method 'aplicar'


## 10. LSP: sustitución de Liskov

*Principio de sustitución de Liskov*: un objeto de una subclase debe poder usarse donde se espera la clase base sin romper su contrato. Escribir una clase `Ave` cuyo método `volar(metros)` devuelve la altura alcanzada y una subclase `Pinguino` que lanza una excepción al volar, y ver cómo falla una función que suelta una bandada. Rediseñar la jerarquía con `AveVoladora` y `AveNadadora` y un método común `desplazarse`, y escribir una prueba de contrato que pasen todas las subclases.

In [16]:
class Ave:
    def __init__(self, nombre):
        self.nombre = nombre

    def volar(self, metros):
        return metros


class PinzonAzul(Ave): ...


class Guirre(Ave):
    def volar(self, metros):
        return min(metros, 2500)


class Pinguino(Ave):
    def volar(self, metros):
        raise RuntimeError(f"{self.nombre} no puede volar")


def cumple_contrato_volar(ave):
    try:
        return 0 < ave.volar(100) <= 100
    except Exception:
        return False


bandada = [PinzonAzul("pinzón azul del Teide"), Guirre("guirre de Fuerteventura"), Pinguino("pingüino del Loro Parque")]
for ave in bandada:
    print(f"{type(ave).__name__:<11} cumple el contrato de Ave: {cumple_contrato_volar(ave)}")
try:
    print([ave.volar(300) for ave in bandada])
except RuntimeError as e:
    print("La bandada falla:", e)
assert [cumple_contrato_volar(a) for a in bandada] == [True, True, False]

PinzonAzul  cumple el contrato de Ave: True
Guirre      cumple el contrato de Ave: True
Pinguino    cumple el contrato de Ave: False
La bandada falla: pingüino del Loro Parque no puede volar


In [17]:
class Ave(ABC):
    def __init__(self, nombre):
        self.nombre = nombre

    @abstractmethod
    def desplazarse(self, metros):
        """Devuelve una descripción del desplazamiento de `metros`."""


class AveVoladora(Ave):
    techo = 10_000

    def volar(self, metros):
        return min(metros, self.techo)

    def desplazarse(self, metros):
        return f"{self.nombre} vuela a {self.volar(metros)} m"


class AveNadadora(Ave):
    def nadar(self, metros):
        return metros

    def desplazarse(self, metros):
        return f"{self.nombre} nada {self.nadar(metros)} m"


class PinzonAzul(AveVoladora): ...


class Guirre(AveVoladora):
    techo = 2500


class Pinguino(AveNadadora): ...


def cumple_contrato(ave):
    texto = ave.desplazarse(100)
    vuela_bien = not isinstance(ave, AveVoladora) or 0 < ave.volar(100) <= 100
    return isinstance(texto, str) and ave.nombre in texto and vuela_bien


aves = [PinzonAzul("pinzón azul del Teide"), Guirre("guirre de Fuerteventura"), Pinguino("pingüino del Loro Parque")]
for ave in aves:
    print(ave.desplazarse(300))
voladoras = [a for a in aves if isinstance(a, AveVoladora)]
print("Bandada de voladoras:", [a.volar(3000) for a in voladoras])
assert all(map(cumple_contrato, aves)) and not hasattr(Pinguino, "volar")

pinzón azul del Teide vuela a 300 m
guirre de Fuerteventura vuela a 300 m
pingüino del Loro Parque nada 300 m
Bandada de voladoras: [3000, 2500]


## 11. ISP: segregación de interfaces

*Principio de segregación de interfaces*: ninguna clase debe verse obligada a depender de métodos que no usa. Escribir una interfaz `Multifuncion` con `imprimir`, `escanear` y `enviar_fax`, y ver cómo una impresora básica tiene que implementar métodos que solo lanzan `NotImplementedError`. Dividirla en tres protocolos pequeños (`Imprime`, `Escanea`, `EnviaFax`) y crear una impresora básica, una con escáner y una multifunción que implementen solo lo que saben hacer. Escanear un lote de documentos con todos los aparatos que pueden hacerlo, sin errores.

In [18]:
from typing import Protocol, runtime_checkable


class Multifuncion(ABC):
    @abstractmethod
    def imprimir(self, documento): ...

    @abstractmethod
    def escanear(self, documento): ...

    @abstractmethod
    def enviar_fax(self, documento, numero): ...


class ImpresoraBasicaForzada(Multifuncion):
    def imprimir(self, documento):
        return f"impreso: {documento}"

    def escanear(self, documento):
        raise NotImplementedError("esta impresora no escanea")

    def enviar_fax(self, documento, numero):
        raise NotImplementedError("esta impresora no envía fax")


try:
    ImpresoraBasicaForzada().escanear("factura.pdf")
except NotImplementedError as e:
    print("Interfaz gorda:", e)

Interfaz gorda: esta impresora no escanea


In [19]:
@runtime_checkable
class Imprime(Protocol):
    def imprimir(self, documento) -> str: ...


@runtime_checkable
class Escanea(Protocol):
    def escanear(self, documento) -> str: ...


@runtime_checkable
class EnviaFax(Protocol):
    def enviar_fax(self, documento, numero) -> str: ...


class ImpresoraBasica:
    def imprimir(self, documento):
        return f"impreso: {documento}"


class ImpresoraEscaner(ImpresoraBasica):
    def escanear(self, documento):
        return f"escaneado: {documento}"


class EquipoMultifuncion(ImpresoraEscaner):
    def enviar_fax(self, documento, numero):
        return f"fax de {documento} a {numero}"


aparatos = [ImpresoraBasica(), ImpresoraEscaner(), EquipoMultifuncion()]
print(f"{'aparato':<18}{'Imprime':>9}{'Escanea':>9}{'EnviaFax':>10}")
for a in aparatos:
    print(f"{type(a).__name__:<18}" + "".join(f"{isinstance(a, p)!s:>{w}}"
                                            for p, w in ((Imprime, 9), (Escanea, 9), (EnviaFax, 10))))


def escanear_lote(documentos, aparatos):
    escaneres = [a for a in aparatos if isinstance(a, Escanea)]
    return [escaneres[i % len(escaneres)].escanear(d) for i, d in enumerate(documentos)]


resultado = escanear_lote(["dni.pdf", "contrato.pdf", "recibo.pdf"], aparatos)
print(resultado)
assert len(resultado) == 3 and not isinstance(ImpresoraBasica(), Escanea)
for a in aparatos:
    for protocolo, llamada in ((Imprime, lambda x: x.imprimir("d")), (Escanea, lambda x: x.escanear("d")),
                               (EnviaFax, lambda x: x.enviar_fax("d", "928000000"))):
        if isinstance(a, protocolo):
            assert isinstance(llamada(a), str)

aparato             Imprime  Escanea  EnviaFax
ImpresoraBasica        True    False     False
ImpresoraEscaner       True     True     False
EquipoMultifuncion     True     True      True
['escaneado: dni.pdf', 'escaneado: contrato.pdf', 'escaneado: recibo.pdf']


## 12. DIP: inversión de dependencias

*Principio de inversión de dependencias*: los módulos de alto nivel no deben depender de detalles concretos, sino de abstracciones. Escribir un `ServicioPedidosAcoplado` que crea dentro su propio cliente de correo y comprobar que no se puede probar sin servidor. Rediseñarlo con un protocolo `CanalNotificacion` y canales de correo, SMS y *push* que se inyectan en el constructor; si un canal falla, el servicio prueba con el siguiente. Probarlo con dobles de prueba: un canal falso que guarda los mensajes y otro que siempre falla.

In [20]:
class ClienteSMTP:
    def enviar(self, destino, texto):
        raise ConnectionError("no hay servidor SMTP en este entorno")


class ServicioPedidosAcoplado:
    def __init__(self):
        self.correo = ClienteSMTP()

    def confirmar(self, pedido, cliente):
        self.correo.enviar(cliente["email"], f"Pedido {pedido} confirmado")


try:
    ServicioPedidosAcoplado().confirmar("P-7", {"email": "ana@example.com"})
except ConnectionError as e:
    print("Diseño acoplado, imposible de probar sin red:", e)

Diseño acoplado, imposible de probar sin red: no hay servidor SMTP en este entorno


In [21]:
class CanalNotificacion(Protocol):
    nombre: str

    def enviar(self, cliente, texto) -> None: ...


class CanalEmail:
    nombre = "email"

    def __init__(self):
        self.bandeja = []

    def enviar(self, cliente, texto):
        self.bandeja.append((cliente["email"], f"Asunto: Tu pedido\n\n{texto}"))


class CanalSMS:
    nombre = "sms"

    def __init__(self):
        self.enviados = []

    def enviar(self, cliente, texto):
        if "movil" not in cliente:
            raise ValueError("el cliente no tiene móvil")
        self.enviados.append((cliente["movil"], texto[:160]))


class CanalPush:
    nombre = "push"

    def __init__(self):
        self.enviados = []

    def enviar(self, cliente, texto):
        self.enviados.append((cliente["dispositivo"], texto))


class ServicioPedidos:
    def __init__(self, canales: list[CanalNotificacion]):
        self.canales = canales

    def confirmar(self, pedido, cliente):
        texto = f"Pedido {pedido} confirmado"
        for canal in self.canales:
            try:
                canal.enviar(cliente, texto)
                return canal.nombre
            except Exception:
                continue
        return None


class CanalFalso:
    nombre = "falso"

    def __init__(self):
        self.mensajes = []

    def enviar(self, cliente, texto):
        self.mensajes.append((cliente["nombre"], texto))


class CanalQueFalla:
    nombre = "roto"

    def enviar(self, cliente, texto):
        raise ConnectionError("caído")


falso = CanalFalso()
assert ServicioPedidos([falso]).confirmar("P-1", {"nombre": "Ana"}) == "falso"
assert falso.mensajes == [("Ana", "Pedido P-1 confirmado")]
assert ServicioPedidos([CanalQueFalla(), falso]).confirmar("P-2", {"nombre": "Ana"}) == "falso"
assert ServicioPedidos([CanalQueFalla()]).confirmar("P-3", {"nombre": "Ana"}) is None
print("Pruebas con dobles superadas:", falso.mensajes)

sms, email, push = CanalSMS(), CanalEmail(), CanalPush()
servicio = ServicioPedidos([sms, email, push])
clientes = [{"nombre": "Ana", "email": "ana@example.com", "movil": "600000001"},
            {"nombre": "Kevin", "email": "kevin@example.com"}]
for i, cliente in enumerate(clientes, 1):
    print(f"Pedido P-{i} a {cliente['nombre']}: por {servicio.confirmar(f'P-{i}', cliente)}")
assert len(sms.enviados) == 1 and len(email.bandeja) == 1 and push.enviados == []

Pruebas con dobles superadas: [('Ana', 'Pedido P-1 confirmado'), ('Ana', 'Pedido P-2 confirmado')]
Pedido P-1 a Ana: por sms
Pedido P-2 a Kevin: por email


## 13. Máquina expendedora

Una máquina vende productos con código, precio (en céntimos) y existencias, y acepta monedas de 5, 10, 20 y 50 céntimos y de 1 y 2 €. Escribir `cambio_minimo(cantidad, existencias)`, que devuelva el cambio con el menor número de monedas usando solo las que hay en la caja (o `None` si no se puede), y comprobarlo contra fuerza bruta: con monedas limitadas, coger siempre la mayor puede fallar. Después crear `MaquinaExpendedora` con `insertar`, `seleccionar`, `cancelar` y `reponer_monedas`, y aplicarle una secuencia fija de operaciones comprobando que el dinero cuadra al final.

In [22]:
import random
from collections import Counter
from itertools import product

MONEDAS = (200, 100, 50, 20, 10, 5)


def cambio_minimo(cantidad, existencias):
    mejor = [None] * (cantidad + 1)
    mejor[0] = ()
    for moneda, n in existencias.items():
        for _ in range(n):
            for c in range(cantidad, moneda - 1, -1):
                previo = mejor[c - moneda]
                if previo is not None and (mejor[c] is None or len(previo) + 1 < len(mejor[c])):
                    mejor[c] = previo + (moneda,)
    return None if mejor[cantidad] is None else Counter(mejor[cantidad])


def cambio_voraz(cantidad, existencias):
    cambio = Counter()
    for moneda in sorted(existencias, reverse=True):
        cambio[moneda] = min(existencias[moneda], cantidad // moneda)
        cantidad -= cambio[moneda] * moneda
    return +cambio if cantidad == 0 else None


def minimo_fuerza_bruta(cantidad, existencias):
    monedas = list(existencias)
    sumas = [sum(cuantas) for cuantas in product(*(range(existencias[m] + 1) for m in monedas))
             if sum(c * m for c, m in zip(cuantas, monedas)) == cantidad]
    return min(sumas, default=None)


caja = {50: 1, 20: 3, 10: 0}
print("Cambio de 60 con", caja, "| voraz:", cambio_voraz(60, caja), "| mínimo:", dict(cambio_minimo(60, caja)))
assert cambio_voraz(60, caja) is None and cambio_minimo(60, caja) == Counter({20: 3})

random.seed(13)
for _ in range(150):
    existencias = {m: random.randint(0, 3) for m in MONEDAS}
    cantidad = 5 * random.randint(0, 80)
    cambio = cambio_minimo(cantidad, existencias)
    esperado = minimo_fuerza_bruta(cantidad, existencias)
    assert (cambio is None) == (esperado is None)
    if cambio is not None:
        assert cambio.total() == esperado and sum(m * n for m, n in cambio.items()) == cantidad
        assert all(cambio[m] <= existencias[m] for m in cambio)

Cambio de 60 con {50: 1, 20: 3, 10: 0} | voraz: None | mínimo: {20: 3}


In [23]:
def euros(centimos):
    return f"{centimos / 100:.2f} €".replace(".", ",")


def valor(monedas):
    return sum(m * n for m, n in monedas.items())


class MaquinaExpendedora:
    def __init__(self, productos, caja):
        self.productos = {codigo: dict(nombre=n, precio=p, stock=s) for codigo, (n, p, s) in productos.items()}
        self.caja = Counter(caja)
        self.credito = Counter()
        self.ventas = 0
        self.devuelto = 0

    def insertar(self, moneda):
        if moneda not in MONEDAS:
            return f"moneda de {moneda} céntimos rechazada"
        self.credito[moneda] += 1
        return f"crédito: {euros(valor(self.credito))}"

    def cancelar(self):
        devuelto = valor(self.credito)
        self.devuelto += devuelto
        self.credito = Counter()
        return f"devueltos {euros(devuelto)}"

    def reponer_monedas(self, monedas):
        self.caja.update(monedas)
        return f"repuestas monedas por {euros(valor(monedas))}"

    def seleccionar(self, codigo):
        producto = self.productos.get(codigo)
        if producto is None:
            return f"{codigo} no existe"
        if producto["stock"] == 0:
            return f"{producto['nombre']} agotado"
        pagado = valor(self.credito)
        if pagado < producto["precio"]:
            return f"faltan {euros(producto['precio'] - pagado)} para {producto['nombre']}"
        disponible = self.caja + self.credito
        cambio = cambio_minimo(pagado - producto["precio"], disponible)
        if cambio is None:
            return f"sin cambio para {euros(pagado - producto['precio'])}: introduzca el importe exacto o cancele"
        self.caja = disponible - cambio
        self.credito = Counter()
        producto["stock"] -= 1
        self.ventas += producto["precio"]
        self.devuelto += valor(cambio)
        monedas = " + ".join(f"{n}×{euros(m)}" for m, n in sorted(cambio.items(), reverse=True)) or "nada"
        return f"sale {producto['nombre']}; cambio: {monedas}"


productos = {"A1": ("Agua", 90, 3), "A2": ("Zumo de mango", 160, 1),
             "B1": ("Bocadillo de pata", 285, 2), "B2": ("Barrita de gofio", 135, 0)}
caja_inicial = {100: 2, 50: 1, 20: 3, 5: 2}
maquina = MaquinaExpendedora(productos, caja_inicial)
operaciones = [
    ("insertar", 100), ("seleccionar", "A1"),
    ("insertar", 200), ("seleccionar", "A2"), ("seleccionar", "A2"),
    ("insertar", 2), ("seleccionar", "B2"), ("seleccionar", "C9"),
    ("insertar", 50), ("seleccionar", "A1"), ("insertar", 50), ("seleccionar", "A1"), ("cancelar",),
    ("reponer_monedas", {10: 5, 5: 5}),
    ("insertar", 200), ("insertar", 100), ("seleccionar", "B1"),
    ("insertar", 50), ("insertar", 20), ("insertar", 20), ("seleccionar", "A1"),
]
insertado = repuesto = 0
for nombre, *argumentos in operaciones:
    if nombre == "insertar" and argumentos[0] in MONEDAS:
        insertado += argumentos[0]
    if nombre == "reponer_monedas":
        repuesto += valor(argumentos[0])
    print(f"{nombre:<16}{str(argumentos[0]) if argumentos else '':<18}{getattr(maquina, nombre)(*argumentos)}")

print(f"\nVentas: {euros(maquina.ventas)} | caja: {euros(valor(maquina.caja))} | stock:",
      {c: p["stock"] for c, p in maquina.productos.items()})
assert not maquina.credito
assert valor(maquina.caja) == valor(caja_inicial) + repuesto + insertado - maquina.devuelto
assert valor(maquina.caja) - valor(caja_inicial) - repuesto == maquina.ventas == 90 + 160 + 285 + 90
assert [p["stock"] for p in maquina.productos.values()] == [1, 0, 1, 0]

insertar        100               crédito: 1,00 €
seleccionar     A1                sale Agua; cambio: 2×0,05 €
insertar        200               crédito: 2,00 €
seleccionar     A2                sale Zumo de mango; cambio: 2×0,20 €
seleccionar     A2                Zumo de mango agotado
insertar        2                 moneda de 2 céntimos rechazada
seleccionar     B2                Barrita de gofio agotado
seleccionar     C9                C9 no existe
insertar        50                crédito: 0,50 €
seleccionar     A1                faltan 0,40 € para Agua
insertar        50                crédito: 1,00 €
seleccionar     A1                sin cambio para 0,10 €: introduzca el importe exacto o cancele
cancelar                          devueltos 1,00 €
reponer_monedas {10: 5, 5: 5}     repuestas monedas por 0,75 €
insertar        200               crédito: 2,00 €
insertar        100               crédito: 3,00 €
seleccionar     B1                sale Bocadillo de pata; cambio: 1×0,1